# 01 — Prepare Canonical Inputs: Mouse Brain 1.3M

Downloads and filters the 10X 1.3M mouse brain dataset into canonical h5ad + allowlists.

**Run on Libra (CPU is fine for this step)**

In [ ]:
import os, json, re, time
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

print(f"Scanpy version: {sc.__version__}")

## Download dataset (run once)

In [ ]:
os.makedirs("data", exist_ok=True)
h5_path = "data/1M_neurons_filtered_gene_bc_matrices_h5.h5"
if not os.path.exists(h5_path):
    print("Downloading 1.3M mouse brain dataset (~3.9 GB)...")
    !wget -P data/ https://s3-us-west-2.amazonaws.com/10x.files/samples/cell/1M_neurons/1M_neurons_filtered_gene_bc_matrices_h5.h5
else:
    print(f"Dataset already exists: {h5_path}")
print(f"File size: {os.path.getsize(h5_path) / 1e9:.2f} GB")

## Load config

In [ ]:
with open("benchmark_config.json") as f:
    cfg = json.load(f)

gcfg = cfg["global"]
dcfg = cfg["datasets"]["mouse_brain_1m"]
print(json.dumps(dcfg, indent=2))

## Load raw data

In [ ]:
t0 = time.time()
adata = sc.read_10x_h5(dcfg["input_path"])
adata.var_names = pd.Index(adata.var_names.astype(str))
adata.var_names_make_unique()
print(f"Loaded in {time.time()-t0:.1f}s")
print(f"Raw matrix: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

## Preserve raw counts

In [ ]:
if sparse.issparse(adata.X):
    adata.X = adata.X.tocsr()
    adata.layers["counts"] = adata.X.copy()
else:
    adata.X = np.asarray(adata.X)
    adata.layers["counts"] = adata.X.copy()
print("Counts layer stored.")

## QC annotation

Mouse genes use lowercase `mt-` prefix, but our QC code uppercases first, so `MT-` matching works.

In [ ]:
RIBO_RE = re.compile(r"^RP[SL]", flags=re.IGNORECASE)
HB_RE = re.compile(r"^HB(?!P)", flags=re.IGNORECASE)

var_upper = pd.Index(adata.var_names.astype(str)).str.upper()
adata.var["mt"] = var_upper.str.startswith("MT-")
adata.var["ribo"] = [bool(RIBO_RE.match(g)) for g in var_upper]
adata.var["hb"] = [bool(HB_RE.match(g)) for g in var_upper]

sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"],
    inplace=True, log1p=False, percent_top=None,
)
print(f"MT genes found: {adata.var['mt'].sum()}")
print(f"Median genes/cell: {adata.obs['n_genes_by_counts'].median():.0f}")
print(f"Median MT%: {adata.obs['pct_counts_mt'].median():.2f}%")

## Filter

In [ ]:
before = (adata.n_obs, adata.n_vars)
print(f"Before filtering: {before[0]:,} cells x {before[1]:,} genes")

# 1) Gene filter
sc.pp.filter_genes(adata, min_cells=dcfg["min_cells"])
print(f"After gene filter (min_cells={dcfg['min_cells']}): {adata.n_obs:,} cells x {adata.n_vars:,} genes")

# 2) Cell filter
cell_mask = (
    (adata.obs["n_genes_by_counts"] >= dcfg["min_genes"]) &
    (adata.obs["n_genes_by_counts"] <= dcfg["max_genes"])
)
if dcfg["max_pct_mt"] is not None:
    cell_mask &= adata.obs["pct_counts_mt"] < dcfg["max_pct_mt"]
adata = adata[cell_mask].copy()

print(f"After cell filter: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

## Write outputs

In [ ]:
os.makedirs("write", exist_ok=True)

t0 = time.time()
adata.write_h5ad(dcfg["canonical_h5ad"], compression="gzip")
print(f"Wrote {dcfg['canonical_h5ad']} in {time.time()-t0:.1f}s")

with open(dcfg["canonical_cells_txt"], "w") as f:
    for bc in adata.obs_names.astype(str):
        f.write(f"{bc}\n")

with open(dcfg["canonical_genes_txt"], "w") as f:
    for gene in adata.var_names.astype(str):
        f.write(f"{gene}\n")

summary = {
    "dataset": dcfg["name"],
    "before": {"n_cells": before[0], "n_genes": before[1]},
    "after": {"n_cells": int(adata.n_obs), "n_genes": int(adata.n_vars)},
    "filters": {
        "min_cells": dcfg["min_cells"],
        "min_genes": dcfg["min_genes"],
        "max_genes": dcfg["max_genes"],
        "max_pct_mt": dcfg["max_pct_mt"],
    },
}
with open(dcfg["canonical_summary_json"], "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nCanonical dataset: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"Files written:")
print(f"  {dcfg['canonical_h5ad']}")
print(f"  {dcfg['canonical_cells_txt']}")
print(f"  {dcfg['canonical_genes_txt']}")
print(f"  {dcfg['canonical_summary_json']}")